# DeepGuard AI — Notebook 1 · Datasets & Training

**Team Quad Squad · IEEE AIML-02 Deepfake Detection**

This notebook consolidates every training-related file for reproducibility.

## Datasets used

| Dataset | Source | Size | Content | Role |
|---|---|---|---|---|
| **Hemg/deepfake-and-real-images** | HuggingFace | 1.8 GB | 95k real portraits + 95k StyleGAN faces (256x256) | Primary training (v1, v2, v3) |
| **andrew-bitmind/ffhq-256_stable-diffusion-xl_training_faces** | HuggingFace | 1.45 GB | 24k SD-XL generated faces | Domain expansion (v2, v3) |
| **dragonintelligence/CIFAKE-image-dataset** | HuggingFace | 50 MB | 60k real CIFAR-10 + 60k SD 1.4 generated (32x32) | Broader AI-image signal (v3) |
| **saurabhbagchi/deepfake-image-detection** | Kaggle | 476 MB | 547 fake + 436 real face images | Extra fine-tune (v4) |
| **acroitoru/social_media_deepfakes** | HuggingFace | 2.9 GB | 201 fake + 90 real .mp4 (social-media) | Blurry / multi-face video-frame training (v4) |

## Pretrained models (downloaded)

- **YOLOv8n-face** (arnabdhar/YOLOv8-Face-Detection, ~6 MB) — face detection
- **YuNet face detector** (OpenCV zoo, 227 KB) — fallback face detection
- **Ateeqq/ai-vs-human-image-detector** (355 MB) — retained as an optional AI-generation signal; excluded from the deepfake verdict
- **EfficientNet-B0** (torchvision, ImageNet init) — spatial backbone we fine-tuned

## Held-out results

> **Important:** these random same-source splits measure in-dataset fit, not generalization to unseen face-swap methods, identities, compression, or video sources. They are not forensic validation.

| Model | Trained on | Val AUC | Test AUC |
|---|---|---|---|
| v1 (`model_best.pt`) | Hemg only | 0.9991 | **0.9989** |
| v2 (`model_v2_best.pt`) | v1 + SD-XL + moderate aug | 0.9991 | -- |
| v3 (`model_v3_best.pt`) | v2 + CIFAKE + heavy aug | 0.9868 | -- |
| v4 (`model_v4_best.pt`) | v3 + Kaggle + acroitoru frames | (final run) | -- |


## `train.py` — original EfficientNet-B0 fine-tune on Hemg (v1)

In [ ]:
"""
Deepfake image classifier — fine-tune EfficientNet-B0 on HF Hemg/deepfake-and-real-images.
Loads directly from Parquet shards. AMP + channels_last for speed on the 4060.

Label convention (INTERNAL, in tensor form):
    0 = REAL   (source label: 1)
    1 = FAKE   (source label: 0)
    -> we set  y = 1 - source_label   so 1 = FAKE (positive class = "manipulated")

Usage:
    python train.py                  # trains, saves model_best.pt + model_final.pt
    python train.py --smoke          # 400 samples, 1 epoch (sanity)
    python train.py --epochs 4       # override epoch count
"""
import argparse, io, os, glob, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.amp import autocast, GradScaler

# ---- config ----
DATA_GLOB = r"C:\dl\deepfake\data\hf\data\train-*.parquet"
IMG_SIZE  = 224
BATCH     = 96
EPOCHS    = 4
LR        = 2e-4
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 2       # Windows-safe; parquet is in-memory anyway
SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class ParquetImageDataset(Dataset):
    """Reads image bytes from a list of parquet shards, decodes with PIL."""
    def __init__(self, parquet_paths, indices=None, transform=None):
        # Load all shards fully in RAM — 190k rows * ~15KB/row ≈ 3 GB.
        # Trades RAM for speed (no per-batch parquet open/close).
        frames = [pq.read_table(p, columns=["image", "label"]).to_pandas()
                  for p in parquet_paths]
        self.df = pd.concat(frames, ignore_index=True)
        self.transform = transform
        # invert label so 1 = FAKE
        self.labels = (1 - self.df["label"].to_numpy()).astype(np.int64)
        self.idx = indices if indices is not None else np.arange(len(self.df))

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        row = self.df.iloc[int(self.idx[i])]
        img = Image.open(io.BytesIO(row["image"]["bytes"])).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        y = self.labels[int(self.idx[i])]
        return img, y


def make_splits(n_total, val_frac=0.10, test_frac=0.10, seed=SEED):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_total)
    n_val  = int(n_total * val_frac)
    n_test = int(n_total * test_frac)
    return perm[n_val + n_test:], perm[:n_val], perm[n_val:n_val + n_test]


def build_model():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(0.3, inplace=True),
        nn.Linear(in_f, 1),
    )
    return m


def _rank_auc(y_true, y_score):
    """Pure-numpy ROC-AUC via rank formula. Handles ties OK for our purposes."""
    y_true = np.asarray(y_true); y_score = np.asarray(y_score)
    order = np.argsort(y_score, kind="stable")
    ranks = np.empty_like(order, dtype=np.float64); ranks[order] = np.arange(1, len(order)+1)
    pos = ranks[y_true == 1]; n_pos = len(pos); n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0: return float("nan")
    return float((pos.sum() - n_pos*(n_pos+1)/2) / (n_pos*n_neg))


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        with autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            logit = model(x).squeeze(1)
        probs.append(torch.sigmoid(logit).float().cpu().numpy())
        ys.append(y.numpy())
    probs = np.concatenate(probs); ys = np.concatenate(ys)
    auc = _rank_auc(ys, probs)
    acc = float(((probs > 0.5).astype(int) == ys).mean())
    return auc, acc, probs, ys


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--epochs", type=int, default=EPOCHS)
    ap.add_argument("--data-glob", default=DATA_GLOB)
    args = ap.parse_args()

    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    paths = sorted(glob.glob(args.data_glob))
    assert paths, f"no parquet found at {args.data_glob}"
    print(f"[cfg] device={DEVICE}  shards={len(paths)}  epochs={args.epochs}")

    print("[data] loading parquet into memory (~30s)...")
    t = time.time()
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.1, 0.1, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    # one shared big dataset that owns the dataframe, then per-split views
    base_train = ParquetImageDataset(paths, transform=train_tf)
    base_eval  = ParquetImageDataset(paths, transform=eval_tf)
    # cheap trick: reuse the dataframe object across the two, drop extra RAM
    base_eval.df = base_train.df; base_eval.labels = base_train.labels

    n = len(base_train.df)
    tr_idx, va_idx, te_idx = make_splits(n)
    if args.smoke:
        tr_idx = tr_idx[:400]; va_idx = va_idx[:200]; te_idx = te_idx[:200]
        args.epochs = 1

    train_ds = ParquetImageDataset(paths, indices=tr_idx, transform=train_tf)
    val_ds   = ParquetImageDataset(paths, indices=va_idx, transform=eval_tf)
    test_ds  = ParquetImageDataset(paths, indices=te_idx, transform=eval_tf)
    # share the pre-loaded dataframe so we don't reload from disk
    for ds in (train_ds, val_ds, test_ds):
        ds.df = base_train.df; ds.labels = base_train.labels

    print(f"[data] load done in {time.time()-t:.1f}s  "
          f"train={len(train_ds)}  val={len(val_ds)}  test={len(te_idx)}  "
          f"pos_rate(train)={base_train.labels[tr_idx].mean():.3f}")

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=(NUM_WORKERS > 0))
    val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=(NUM_WORKERS > 0))
    test_loader  = DataLoader(test_ds, batch_size=BATCH, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=(NUM_WORKERS > 0))

    model = build_model().to(DEVICE, memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    loss_fn = nn.BCEWithLogitsLoss()
    scaler = GradScaler(enabled=(DEVICE == "cuda"))

    best_auc = -1.0
    t0 = time.time()
    for ep in range(1, args.epochs + 1):
        model.train()
        running = 0.0; n_seen = 0
        t_ep = time.time()
        for step, (x, y) in enumerate(train_loader, 1):
            x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
            y = y.float().to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                logit = model(x).squeeze(1)
                loss  = loss_fn(logit, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            running += loss.item() * x.size(0); n_seen += x.size(0)
            if step % 50 == 0:
                print(f"  ep{ep} step {step}/{len(train_loader)}  "
                      f"loss={running/n_seen:.4f}  "
                      f"lr={opt.param_groups[0]['lr']:.2e}  "
                      f"elapsed={time.time()-t_ep:.0f}s", flush=True)
        sched.step()
        val_auc, val_acc, _, _ = evaluate(model, val_loader)
        print(f"[ep{ep}] train_loss={running/n_seen:.4f}  "
              f"val_auc={val_auc:.4f}  val_acc={val_acc:.4f}  "
              f"epoch_time={time.time()-t_ep:.0f}s  total={time.time()-t0:.0f}s",
              flush=True)

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save({"model": model.state_dict(),
                        "classes": ["REAL", "FAKE"],  # index = label
                        "val_auc": val_auc,
                        "img_size": IMG_SIZE,
                        "arch": "efficientnet_b0"},
                       "model_best.pt")
            print(f"  -> new best (auc={val_auc:.4f}), saved model_best.pt")

    torch.save({"model": model.state_dict(), "classes": ["REAL", "FAKE"],
                "val_auc": best_auc, "img_size": IMG_SIZE,
                "arch": "efficientnet_b0"},
               "model_final.pt")
    total = time.time() - t0
    print(f"\n[done] best_val_auc={best_auc:.4f}  total_train_time={total:.0f}s")

    print("\n[eval] scoring test split...")
    test_auc, test_acc, _, _ = evaluate(model, test_loader)
    print(f"[test] auc={test_auc:.4f}  acc={test_acc:.4f}")

    Path("training_result.json").write_text(json.dumps({
        "best_val_auc": best_auc,
        "test_auc": test_auc,
        "test_acc": test_acc,
        "epochs": args.epochs,
        "seconds": total,
        "n_train": len(train_ds),
        "n_val": len(val_ds),
        "n_test": len(test_ds),
    }, indent=2))


if __name__ == "__main__":
    main()


## `train_v2.py` — combined dataset training (Hemg + SD-XL)

In [ ]:
"""
train_v2.py — Retrain on COMBINED datasets with heavy augmentation.

Datasets:
  Hemg/deepfake-and-real-images   (StyleGAN + real faces, 190k)   label from parquet
  bitmind/ffhq-256__SD-XL         (Stable-Diffusion faces, ~24k)  all FAKE

Augmentation added vs train.py v1:
  - RandomResizedCrop (0.7-1.0)
  - Stronger ColorJitter
  - Gaussian blur (occasional)
  - JPEG compression sim (via encoding trick)
  - Random rotation ±10 deg

Label convention (internal):  1 = FAKE, 0 = REAL
"""
import argparse, io, glob, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms, models
from torch.amp import autocast, GradScaler

HEMG_GLOB = r"C:\dl\deepfake\data\hf\data\train-*.parquet"
SD_GLOB   = r"C:\dl\deepfake\data\sd_faces\random_aug_transforms\train-*.parquet"
IMG_SIZE  = 224
BATCH     = 96
EPOCHS    = 2
LR        = 1e-4     # lower than v1 since we start from ImageNet weights again
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0        # Windows: multi-worker is slower than in-process here
SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# ---- JPEG-compression augmentation (simulates re-uploaded images) ----
class RandomJPEGCompression:
    def __init__(self, p=0.4, quality_range=(30, 90)):
        self.p = p; self.qr = quality_range
    def __call__(self, img):
        if random.random() > self.p:
            return img
        q = random.randint(*self.qr)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).convert("RGB")


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    RandomJPEGCompression(p=0.4, quality_range=(30, 90)),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class HemgDataset(Dataset):
    """Hemg — has explicit label column. y_out = 1 - label (so 1 = FAKE)."""
    def __init__(self, parquet_paths, transform):
        frames = [pq.read_table(p, columns=["image", "label"]).to_pandas()
                  for p in parquet_paths]
        self.df = pd.concat(frames, ignore_index=True)
        self.labels = (1 - self.df["label"].to_numpy()).astype(np.int64)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = Image.open(io.BytesIO(self.df.iloc[i]["image"]["bytes"])).convert("RGB")
        return self.transform(img), int(self.labels[i])


class SDFakeDataset(Dataset):
    """SD-generated faces — all label = 1 (FAKE)."""
    def __init__(self, parquet_paths, transform):
        frames = [pq.read_table(p, columns=["image"]).to_pandas()
                  for p in parquet_paths]
        self.df = pd.concat(frames, ignore_index=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = Image.open(io.BytesIO(self.df.iloc[i]["image"]["bytes"])).convert("RGB")
        return self.transform(img), 1  # FAKE


def make_splits(n, seed=SEED, val_frac=0.10, test_frac=0.10):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    nv, nt = int(n * val_frac), int(n * test_frac)
    return perm[nv+nt:], perm[:nv], perm[nv:nv+nt]


class SubsetWithTF(Dataset):
    def __init__(self, base, indices, transform):
        self.base = base; self.idx = indices; self.tf = transform
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        img = Image.open(io.BytesIO(self.base.df.iloc[int(self.idx[i])]["image"]["bytes"])).convert("RGB")
        # SD ds has no label col
        if hasattr(self.base, "labels"):
            y = int(self.base.labels[int(self.idx[i])])
        else:
            y = 1
        return self.tf(img), y


def build_model():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(in_f, 1))
    return m


def _rank_auc(y, p):
    order = np.argsort(p, kind="stable")
    ranks = np.empty_like(order, dtype=np.float64); ranks[order] = np.arange(1, len(order)+1)
    pos = ranks[y == 1]; n_pos = len(pos); n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0: return float("nan")
    return float((pos.sum() - n_pos*(n_pos+1)/2) / (n_pos*n_neg))


@torch.no_grad()
def eval_loader(model, loader):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        with autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            logit = model(x).squeeze(1)
        probs.append(torch.sigmoid(logit).float().cpu().numpy())
        ys.append(y.numpy())
    probs = np.concatenate(probs); ys = np.concatenate(ys)
    return _rank_auc(ys, probs), float(((probs > 0.5).astype(int) == ys).mean())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=EPOCHS)
    ap.add_argument("--smoke", action="store_true")
    args = ap.parse_args()

    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    print(f"[cfg] device={DEVICE}  epochs={args.epochs}")

    hemg_paths = sorted(glob.glob(HEMG_GLOB))
    sd_paths   = sorted(glob.glob(SD_GLOB))
    assert hemg_paths, "no Hemg parquet"
    print(f"[data] Hemg shards={len(hemg_paths)}  SD shards={len(sd_paths)}")

    t = time.time()
    hemg = HemgDataset(hemg_paths, transform=train_tf)
    sd   = SDFakeDataset(sd_paths, transform=train_tf) if sd_paths else None
    print(f"[data] Hemg={len(hemg)} rows  "
          f"SD={len(sd) if sd else 0} rows  loaded in {time.time()-t:.0f}s")

    # Splits — do them on the combined index space
    # Hemg indices 0..len(hemg)-1, SD indices len(hemg)..len(hemg)+len(sd)-1
    n_total = len(hemg) + (len(sd) if sd else 0)
    tr_idx, va_idx, te_idx = make_splits(n_total)
    if args.smoke:
        tr_idx = tr_idx[:800]; va_idx = va_idx[:200]; te_idx = te_idx[:200]
        args.epochs = 1

    def make_subset(indices, transform):
        # split indices back into (source, local_idx)
        hemg_mask = indices < len(hemg)
        sd_mask   = ~hemg_mask
        parts = []
        if hemg_mask.any():
            parts.append(SubsetWithTF(hemg, indices[hemg_mask], transform))
        if sd is not None and sd_mask.any():
            local = indices[sd_mask] - len(hemg)
            parts.append(SubsetWithTF(sd, local, transform))
        return ConcatDataset(parts) if len(parts) > 1 else parts[0]

    train_ds = make_subset(tr_idx, train_tf)
    val_ds   = make_subset(va_idx, eval_tf)
    test_ds  = make_subset(te_idx, eval_tf)

    # log positive rate in train subset
    def pos_rate(indices):
        y = []
        for i in indices:
            y.append(int(hemg.labels[i]) if i < len(hemg) else 1)
        return float(np.mean(y))
    print(f"[data] train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}  "
          f"pos_rate(train)={pos_rate(tr_idx):.3f}")

    tl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=True,
                    persistent_workers=False)
    vl = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True,
                    persistent_workers=False)
    tel = DataLoader(test_ds, batch_size=BATCH, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=True,
                     persistent_workers=False)

    model = build_model().to(DEVICE, memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    loss_fn = nn.BCEWithLogitsLoss()
    scaler = GradScaler(enabled=(DEVICE == "cuda"))

    best_auc = -1.0; t0 = time.time()
    for ep in range(1, args.epochs + 1):
        model.train()
        running = 0.0; n_seen = 0; t_ep = time.time()
        for step, (x, y) in enumerate(tl, 1):
            x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
            y = y.float().to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                logit = model(x).squeeze(1)
                loss  = loss_fn(logit, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            running += loss.item() * x.size(0); n_seen += x.size(0)
            if step % 50 == 0:
                print(f"  ep{ep} step {step}/{len(tl)}  loss={running/n_seen:.4f}  "
                      f"lr={opt.param_groups[0]['lr']:.2e}  elapsed={time.time()-t_ep:.0f}s",
                      flush=True)
        sched.step()
        auc, acc = eval_loader(model, vl)
        print(f"[ep{ep}] train_loss={running/n_seen:.4f}  val_auc={auc:.4f}  "
              f"val_acc={acc:.4f}  time={time.time()-t_ep:.0f}s  total={time.time()-t0:.0f}s",
              flush=True)
        if auc > best_auc:
            best_auc = auc
            torch.save({"model": model.state_dict(),
                        "classes": ["REAL", "FAKE"],
                        "val_auc": auc, "img_size": IMG_SIZE,
                        "arch": "efficientnet_b0", "version": "v2"},
                       "model_v2_best.pt")
            print(f"  -> new best v2 (auc={auc:.4f}), saved model_v2_best.pt")

    torch.save({"model": model.state_dict(), "classes": ["REAL", "FAKE"],
                "val_auc": best_auc, "img_size": IMG_SIZE,
                "arch": "efficientnet_b0", "version": "v2"},
               "model_v2_final.pt")
    total = time.time() - t0

    print("\n[eval] scoring test split...")
    test_auc, test_acc = eval_loader(model, tel)
    print(f"[test] auc={test_auc:.4f}  acc={test_acc:.4f}")

    Path("training_result_v2.json").write_text(json.dumps({
        "best_val_auc": best_auc, "test_auc": test_auc,
        "test_acc": test_acc, "epochs": args.epochs, "seconds": total,
        "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
    }, indent=2))
    print(f"[done] best_val_auc={best_auc:.4f}  total={total:.0f}s")


if __name__ == "__main__":
    main()


## `finetune_v3.py` — v2 -> v3 with CIFAKE + heavy augmentation

In [ ]:
"""
finetune_v3.py — Retrain from model_v2_best.pt with CIFAKE + heavy augmentation.

Adds ~100k images from CIFAKE (real CIFAR-10 vs SD 1.4 generated) to teach
the model general AI-image fingerprints beyond just faces.

Datasets:
  Hemg  (~30k sampled, both labels — StyleGAN + real faces)
  SD    (24k, all fake — SDXL faces)
  CIFAKE (~40k sampled, both labels — general AI vs real objects)

Aggressive augmentation (JPEG q=30..95, GaussianBlur sigma up to 3.0,
ColorJitter 0.4, RandomAffine, RandomErasing).

Output: model_v3_best.pt
"""
import argparse, io, glob, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms, models
from torch.amp import autocast, GradScaler

HEMG_GLOB   = r"C:\dl\deepfake\data\hf\data\train-*.parquet"
SD_GLOB     = r"C:\dl\deepfake\data\sd_faces\random_aug_transforms\train-*.parquet"
CIFAKE_GLOB = r"C:\dl\deepfake\data\cifake\data\train-*.parquet"
WEIGHTS_IN  = "model_v2_best.pt"
WEIGHTS_OUT = "model_v3_best.pt"
IMG_SIZE  = 224
BATCH     = 96
LR        = 4e-5
EPOCHS    = 1
HEMG_SAMPLE   = 30000
CIFAKE_SAMPLE = 40000
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]


class RandomJPEG:
    def __init__(self, p=0.6, q=(30, 95)): self.p = p; self.q = q
    def __call__(self, img):
        if random.random() > self.p: return img
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(*self.q))
        buf.seek(0)
        return Image.open(buf).convert("RGB")


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(0.8, 1.2)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.15),
    RandomJPEG(p=0.6, q=(30, 95)),
    transforms.RandomApply([transforms.GaussianBlur(3, (0.1, 3.0))], p=0.4),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class ParquetDS(Dataset):
    """
    Generic parquet reader.
      label_map: callable(src_label) -> internal_label   (1=FAKE, 0=REAL)
      constant_label: use when parquet has no label column
    """
    def __init__(self, paths, indices=None, label_map=None,
                 constant_label=None, transform=None):
        cols = ["image"] if constant_label is not None else ["image", "label"]
        frames = [pq.read_table(p, columns=cols).to_pandas() for p in paths]
        self.df = pd.concat(frames, ignore_index=True)
        self.constant_label = constant_label
        if constant_label is None:
            src = self.df["label"].to_numpy()
            self.labels = np.array([label_map(int(v)) for v in src], dtype=np.int64)
        self.idx = indices if indices is not None else np.arange(len(self.df))
        self.tf = transform

    def __len__(self): return len(self.idx)

    def __getitem__(self, i):
        j = int(self.idx[i])
        img = Image.open(io.BytesIO(self.df.iloc[j]["image"]["bytes"])).convert("RGB")
        y = self.constant_label if self.constant_label is not None else int(self.labels[j])
        return self.tf(img), y


def build_model():
    m = models.efficientnet_b0(weights=None)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(in_f, 1))
    return m


def _rank_auc(y, p):
    order = np.argsort(p, kind="stable")
    ranks = np.empty_like(order, dtype=np.float64); ranks[order] = np.arange(1, len(order)+1)
    pos = ranks[y == 1]; n_pos = len(pos); n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0: return float("nan")
    return float((pos.sum() - n_pos*(n_pos+1)/2) / (n_pos*n_neg))


@torch.no_grad()
def eval_loader(model, loader):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        with autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            logit = model(x).squeeze(1)
        probs.append(torch.sigmoid(logit).float().cpu().numpy())
        ys.append(y.numpy())
    probs = np.concatenate(probs); ys = np.concatenate(ys)
    return _rank_auc(ys, probs), float(((probs > 0.5).astype(int) == ys).mean())


def main():
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    print(f"[cfg] device={DEVICE}  loading data...", flush=True)

    hemg_paths   = sorted(glob.glob(HEMG_GLOB))
    sd_paths     = sorted(glob.glob(SD_GLOB))
    cifake_paths = sorted(glob.glob(CIFAKE_GLOB))
    print(f"[data] Hemg shards={len(hemg_paths)}  SD shards={len(sd_paths)}  "
          f"CIFAKE shards={len(cifake_paths)}", flush=True)

    t = time.time()
    # Hemg: label 0 = fake (invert to 1 = FAKE), label 1 = real (invert to 0 = REAL)
    hemg_full   = ParquetDS(hemg_paths,   label_map=lambda l: 1 - l, transform=train_tf)
    # SD: all fake
    sd_full     = ParquetDS(sd_paths,     constant_label=1,          transform=train_tf)
    # CIFAKE: label 0 = REAL (map to 0), label 1 = FAKE (map to 1)  [standard convention]
    cifake_full = ParquetDS(cifake_paths, label_map=lambda l: l,     transform=train_tf) if cifake_paths else None
    print(f"[data] Hemg={len(hemg_full)}  SD={len(sd_full)}  "
          f"CIFAKE={len(cifake_full) if cifake_full else 0}  loaded in {time.time()-t:.0f}s",
          flush=True)

    rng = np.random.default_rng(SEED)

    # Train samples
    hemg_idx = rng.choice(len(hemg_full), size=min(HEMG_SAMPLE, len(hemg_full)), replace=False)
    hemg_train = ParquetDS(hemg_paths, indices=hemg_idx, label_map=lambda l: 1 - l, transform=train_tf)
    hemg_train.df = hemg_full.df; hemg_train.labels = hemg_full.labels

    if cifake_full is not None:
        cif_idx = rng.choice(len(cifake_full), size=min(CIFAKE_SAMPLE, len(cifake_full)), replace=False)
        cif_train = ParquetDS(cifake_paths, indices=cif_idx, label_map=lambda l: l, transform=train_tf)
        cif_train.df = cifake_full.df; cif_train.labels = cifake_full.labels

    train_parts = [hemg_train, sd_full]
    if cifake_full is not None: train_parts.append(cif_train)
    train_ds = ConcatDataset(train_parts)

    # Val: small held-out
    val_hemg_idx = rng.choice(len(hemg_full), size=1500, replace=False)
    val_sd_idx   = rng.choice(len(sd_full),   size=750,  replace=False)
    val_hemg = ParquetDS(hemg_paths, indices=val_hemg_idx, label_map=lambda l: 1 - l, transform=eval_tf)
    val_hemg.df = hemg_full.df; val_hemg.labels = hemg_full.labels
    val_sd   = ParquetDS(sd_paths,   indices=val_sd_idx,   constant_label=1,          transform=eval_tf)
    val_sd.df = sd_full.df
    val_parts = [val_hemg, val_sd]
    if cifake_full is not None:
        val_cif_idx = rng.choice(len(cifake_full), size=1500, replace=False)
        val_cif = ParquetDS(cifake_paths, indices=val_cif_idx, label_map=lambda l: l, transform=eval_tf)
        val_cif.df = cifake_full.df; val_cif.labels = cifake_full.labels
        val_parts.append(val_cif)
    val_ds = ConcatDataset(val_parts)

    print(f"[data] train={len(train_ds)}  val={len(val_ds)}", flush=True)

    tl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
    vl = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

    model = build_model().to(DEVICE, memory_format=torch.channels_last)
    ckpt = torch.load(WEIGHTS_IN, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    print(f"[model] loaded from {WEIGHTS_IN} (prev val_auc={ckpt.get('val_auc'):.4f})", flush=True)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    scaler = GradScaler(enabled=(DEVICE == "cuda"))

    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        model.train()
        running = 0.0; n_seen = 0; t_ep = time.time()
        for step, (x, y) in enumerate(tl, 1):
            x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
            y = y.float().to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                logit = model(x).squeeze(1)
                loss  = loss_fn(logit, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            running += loss.item() * x.size(0); n_seen += x.size(0)
            if step % 25 == 0:
                print(f"  ep{ep} step {step}/{len(tl)}  loss={running/n_seen:.4f}  "
                      f"elapsed={time.time()-t_ep:.0f}s", flush=True)
        auc, acc = eval_loader(model, vl)
        print(f"[ep{ep}] loss={running/n_seen:.4f}  val_auc={auc:.4f}  val_acc={acc:.4f}  "
              f"time={time.time()-t_ep:.0f}s", flush=True)

    torch.save({"model": model.state_dict(), "classes": ["REAL", "FAKE"],
                "val_auc": auc, "img_size": IMG_SIZE,
                "arch": "efficientnet_b0", "version": "v3-cifake"},
               WEIGHTS_OUT)
    total = time.time() - t0
    print(f"[done] val_auc={auc:.4f}  total={total:.0f}s  saved {WEIGHTS_OUT}", flush=True)

    Path("finetune_v3_result.json").write_text(json.dumps({
        "val_auc": auc, "val_acc": acc, "seconds": total,
        "n_train": len(train_ds), "n_val": len(val_ds),
    }, indent=2))


if __name__ == "__main__":
    main()


## `finetune_v4.py` — v3 -> v4 with Kaggle + acroitoru video frames

In [ ]:
"""
finetune_v4.py — fast fine-tune of model_v3 with Kaggle deepfake images added.

Starts from model_v3_best.pt (already generalized), adds:
  - Kaggle saurabhbagchi/deepfake-image-detection train/ set (~800 imgs)
  - Optional: extracted face frames from acroitoru/social_media_deepfakes videos

Small dataset → 1 quick epoch is enough to nudge the model.
"""
import argparse, glob, io, json, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms, models
from torch.amp import autocast, GradScaler

HEMG_GLOB    = r"C:\dl\deepfake\data\hf\data\train-*.parquet"
SD_GLOB      = r"C:\dl\deepfake\data\sd_faces\random_aug_transforms\train-*.parquet"
CIFAKE_GLOB  = r"C:\dl\deepfake\data\cifake\data\train-*.parquet"
KAGGLE_ROOT  = r"C:\dl\deepfake\data\kaggle_deepfake"
ACROITORU    = r"C:\dl\deepfake\data\acroitoru\social_media"
V3_PATH      = "model_v3_best.pt"
OUT_BEST     = "model_v4_best.pt"

IMG_SIZE = 224
BATCH    = 64
EPOCHS   = 1
LR       = 3e-5
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
SEED     = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

import random
class RandomJPEGCompression:
    def __init__(self, p=0.6, q=(25, 90)): self.p = p; self.q = q
    def __call__(self, img):
        if random.random() > self.p: return img
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=random.randint(*self.q)); buf.seek(0)
        return Image.open(buf).convert("RGB")

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    RandomJPEGCompression(p=0.6, q=(25, 90)),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 2.5))], p=0.4),
    transforms.RandomApply([transforms.RandomErasing(p=1.0, value=0)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ParquetDS(Dataset):
    """Parquet with image bytes + optional label."""
    def __init__(self, paths, transform, fixed_label=None, invert_label=False):
        cols = ["image", "label"] if fixed_label is None else ["image"]
        frames = [pq.read_table(p, columns=cols).to_pandas() for p in paths]
        self.df = pd.concat(frames, ignore_index=True)
        self.transform = transform
        if fixed_label is not None:
            self.labels = np.full(len(self.df), fixed_label, dtype=np.int64)
        elif invert_label:
            self.labels = (1 - self.df["label"].to_numpy()).astype(np.int64)
        else:
            self.labels = self.df["label"].to_numpy().astype(np.int64)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = Image.open(io.BytesIO(self.df.iloc[i]["image"]["bytes"])).convert("RGB")
        return self.transform(img), int(self.labels[i])


class FolderDS(Dataset):
    """Reads .jpg/.png from real/ and fake/ subfolders."""
    def __init__(self, root, transform):
        self.transform = transform
        self.samples = []
        for lbl_name, lbl in [("real", 0), ("fake", 1)]:
            for p in Path(root).rglob(f"{lbl_name}/*"):
                if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                    self.samples.append((str(p), lbl))
        print(f"[FolderDS {root}] {len(self.samples)} samples")
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, y = self.samples[i]
        img = Image.open(p).convert("RGB")
        return self.transform(img), int(y)


def build_model():
    m = models.efficientnet_b0(weights=None)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(in_f, 1))
    return m


def _rank_auc(y, p):
    order = np.argsort(p, kind="stable")
    ranks = np.empty_like(order, dtype=np.float64); ranks[order] = np.arange(1, len(order)+1)
    pos = ranks[y == 1]; n_pos = len(pos); n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0: return float("nan")
    return float((pos.sum() - n_pos*(n_pos+1)/2) / (n_pos*n_neg))


@torch.no_grad()
def eval_loader(model, loader):
    model.eval(); probs, ys = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        with autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            probs.append(torch.sigmoid(model(x).squeeze(1)).float().cpu().numpy())
        ys.append(y.numpy())
    probs = np.concatenate(probs); ys = np.concatenate(ys)
    return _rank_auc(ys, probs), float(((probs > 0.5).astype(int) == ys).mean())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--include-acroitoru", action="store_true",
                    help="also add extracted face frames from acroitoru videos")
    ap.add_argument("--epochs", type=int, default=EPOCHS)
    args = ap.parse_args()

    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

    parts = []

    # Hemg (single shard for speed — new-data emphasis)
    hemg = sorted(glob.glob(HEMG_GLOB))[:1]
    if hemg:
        d = ParquetDS(hemg, transform=train_tf, invert_label=True)
        parts.append(d); print(f"[data] Hemg subset: {len(d)}")

    # SD faces — all fake (2 shards for balance)
    sd = sorted(glob.glob(SD_GLOB))[:2]
    if sd:
        d = ParquetDS(sd, transform=train_tf, fixed_label=1)
        parts.append(d); print(f"[data] SD: {len(d)}")

    # CIFAKE
    cif = sorted(glob.glob(CIFAKE_GLOB))
    if cif:
        d = ParquetDS(cif, transform=train_tf, invert_label=False)
        parts.append(d); print(f"[data] CIFAKE: {len(d)}")

    # Kaggle
    if os.path.exists(KAGGLE_ROOT):
        d = FolderDS(KAGGLE_ROOT, transform=train_tf)
        if len(d) > 0:
            parts.append(d); print(f"[data] Kaggle: {len(d)}")

    # Acroitoru (only if flag set and folder exists)
    if args.include_acroitoru and os.path.exists(ACROITORU):
        d = FolderDS(ACROITORU, transform=train_tf)
        if len(d) > 0:
            parts.append(d); print(f"[data] acroitoru frames: {len(d)}")

    if not parts:
        raise RuntimeError("no data")

    ds = ConcatDataset(parts)
    n = len(ds); print(f"[data] TOTAL: {n}")

    # 90/10 split
    idx = np.arange(n); np.random.default_rng(SEED).shuffle(idx)
    val_n = max(500, n // 20); tr_idx = idx[val_n:]; va_idx = idx[:val_n]
    from torch.utils.data import Subset
    tr_ds = Subset(ds, tr_idx.tolist()); va_ds = Subset(ds, va_idx.tolist())

    tl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=True)
    vl = DataLoader(va_ds, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

    m = build_model().to(DEVICE, memory_format=torch.channels_last)
    if os.path.exists(V3_PATH):
        ckpt = torch.load(V3_PATH, map_location=DEVICE)
        m.load_state_dict(ckpt["model"])
        print(f"[model] loaded {V3_PATH}")
    else:
        print(f"[model] {V3_PATH} not found — training from ImageNet init")

    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    scaler = GradScaler(enabled=(DEVICE == "cuda"))
    best_auc = -1.0; t0 = time.time()

    for ep in range(1, args.epochs + 1):
        m.train(); running = 0.0; nn_seen = 0; t_ep = time.time()
        for step, (x, y) in enumerate(tl, 1):
            x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
            y = y.float().to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                loss = loss_fn(m(x).squeeze(1), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            running += loss.item() * x.size(0); nn_seen += x.size(0)
            if step % 25 == 0:
                print(f"  ep{ep} step {step}/{len(tl)}  loss={running/nn_seen:.4f}  "
                      f"elapsed={time.time()-t_ep:.0f}s", flush=True)
        auc, acc = eval_loader(m, vl)
        print(f"[ep{ep}] loss={running/nn_seen:.4f}  val_auc={auc:.4f}  val_acc={acc:.4f}  "
              f"time={time.time()-t_ep:.0f}s", flush=True)
        if auc > best_auc:
            best_auc = auc
            torch.save({"model": m.state_dict(), "classes": ["REAL", "FAKE"],
                        "val_auc": auc, "version": "v4-kaggle+aug"}, OUT_BEST)
            print(f"  -> saved {OUT_BEST}")

    Path("training_result_v4.json").write_text(json.dumps({
        "best_val_auc": best_auc, "seconds": time.time()-t0, "n_train": len(tr_ds),
    }))
    print(f"[done] best={best_auc:.4f} total={time.time()-t0:.0f}s")


if __name__ == "__main__":
    main()


## `extract_video_faces.py` — pull face crops from acroitoru videos

In [ ]:
"""
extract_video_faces.py — pull face crops from acroitoru real/fake videos.

Output:
  C:\\dl\\deepfake\\data\\acroitoru\\social_media\\real\\*.jpg
  C:\\dl\\deepfake\\data\\acroitoru\\social_media\\fake\\*.jpg

For each video: sample ~6 frames, detect faces with YOLO (via faces.py),
save each face crop as a 256x256 JPEG.
"""
import os, sys, glob, time
from pathlib import Path
import cv2
sys.path.insert(0, r"C:\dl\deepfake")
from faces import detect_and_crop, sample_video_frames

SRC = r"C:\dl\deepfake\data\acroitoru\social_media"
OUT = r"C:\dl\deepfake\data\acroitoru\social_media"   # writes into real/ fake/ next to source
FRAMES_PER_VIDEO = 6
OUT_SIZE = 256

def process(label: str):
    src_dir = Path(SRC) / label
    out_dir = Path(OUT) / label
    out_dir.mkdir(parents=True, exist_ok=True)

    videos = sorted(glob.glob(str(src_dir / "*.mp4")))
    print(f"[{label}] {len(videos)} videos found in {src_dir}")
    total_crops = 0; t0 = time.time()
    for vi, vpath in enumerate(videos):
        try:
            frames = sample_video_frames(vpath, fps_sample=2.0)
            # cap to N frames total
            frames = frames[:FRAMES_PER_VIDEO]
            vid_id = Path(vpath).stem
            for fi, (idx, frame) in enumerate(frames):
                if frame is None: continue
                crops = detect_and_crop(frame, out_size=OUT_SIZE, margin=0.28)
                for ci, crop in enumerate(crops[:3]):    # max 3 faces per frame
                    out_path = out_dir / f"{vid_id}_f{idx}_c{ci}.jpg"
                    cv2.imwrite(str(out_path), crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
                    total_crops += 1
        except Exception as e:
            print(f"  err {vpath}: {e}")
        if (vi + 1) % 20 == 0:
            print(f"  [{label}] {vi+1}/{len(videos)}  crops={total_crops}  "
                  f"elapsed={time.time()-t0:.0f}s", flush=True)
    print(f"[{label}] DONE  {total_crops} crops in {time.time()-t0:.0f}s")


if __name__ == "__main__":
    for lbl in ("real", "fake"):
        process(lbl)


## `evaluate.py` — ROC-AUC + classification report on held-out test

In [ ]:
"""
Evaluate a trained deepfake classifier on the held-out test split
of the HF Hemg/deepfake-and-real-images parquet dataset.

Uses the same 42-seed 80/10/10 split as train.py — so the test set is
data the model has NEVER seen during training.

Usage:
    python evaluate.py --weights model_best.pt
"""
import argparse, io, glob, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.amp import autocast

# must match train.py exactly
DATA_GLOB = r"C:\dl\deepfake\data\hf\data\train-*.parquet"
IMG_SIZE  = 224
BATCH     = 128
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ParquetImageDataset(Dataset):
    def __init__(self, parquet_paths, indices=None, transform=None):
        frames = [pq.read_table(p, columns=["image", "label"]).to_pandas()
                  for p in parquet_paths]
        self.df = pd.concat(frames, ignore_index=True)
        self.transform = transform
        self.labels = (1 - self.df["label"].to_numpy()).astype(np.int64)  # 1 = FAKE
        self.idx = indices if indices is not None else np.arange(len(self.df))

    def __len__(self): return len(self.idx)

    def __getitem__(self, i):
        row = self.df.iloc[int(self.idx[i])]
        img = Image.open(io.BytesIO(row["image"]["bytes"])).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[int(self.idx[i])]


def make_splits(n_total, val_frac=0.10, test_frac=0.10, seed=SEED):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_total)
    n_val  = int(n_total * val_frac)
    n_test = int(n_total * test_frac)
    return perm[n_val + n_test:], perm[:n_val], perm[n_val:n_val + n_test]


def build_model():
    m = models.efficientnet_b0(weights=None)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(in_f, 1))
    return m


def _rank_auc(y, p):
    order = np.argsort(p, kind="stable")
    ranks = np.empty_like(order, dtype=np.float64); ranks[order] = np.arange(1, len(order)+1)
    pos = ranks[y == 1]; n_pos = len(pos); n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0: return float("nan")
    return float((pos.sum() - n_pos*(n_pos+1)/2) / (n_pos*n_neg))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", default="model_best.pt")
    ap.add_argument("--data-glob", default=DATA_GLOB)
    args = ap.parse_args()

    ckpt = torch.load(args.weights, map_location=DEVICE)
    print(f"[cfg] weights={args.weights}  classes={ckpt.get('classes')}  "
          f"saved_val_auc={ckpt.get('val_auc'):.4f}")

    paths = sorted(glob.glob(args.data_glob))
    assert paths, f"no parquet at {args.data_glob}"
    print("[data] loading parquet (~15s)...")
    t = time.time()
    base = ParquetImageDataset(paths, transform=eval_tf)
    _, _, te_idx = make_splits(len(base.df))
    test_ds = ParquetImageDataset(paths, indices=te_idx, transform=eval_tf)
    test_ds.df = base.df; test_ds.labels = base.labels
    print(f"[data] test={len(test_ds)}  load={time.time()-t:.1f}s  "
          f"pos_rate={base.labels[te_idx].mean():.3f}")

    model = build_model().to(DEVICE)
    model.load_state_dict(ckpt["model"])
    model.eval()

    loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False,
                        num_workers=2, pin_memory=True, persistent_workers=True)

    t = time.time()
    probs, ys = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            with autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                logit = model(x).squeeze(1)
            probs.append(torch.sigmoid(logit).float().cpu().numpy())
            ys.append(y.numpy())
    probs = np.concatenate(probs); ys = np.concatenate(ys)
    print(f"[infer] {len(probs)} images in {time.time()-t:.1f}s")

    auc = _rank_auc(ys, probs)
    pred = (probs > 0.5).astype(int)
    acc = float((pred == ys).mean())

    def pr(cls):
        tp = int(((pred == cls) & (ys == cls)).sum())
        fp = int(((pred == cls) & (ys != cls)).sum())
        fn = int(((pred != cls) & (ys == cls)).sum())
        p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
        return p, r

    p_real, r_real = pr(0); p_fake, r_fake = pr(1)
    print("\n" + "=" * 60)
    print("             TEST-SET RESULTS  (held-out 19,033 imgs)")
    print("=" * 60)
    print(f"  ROC-AUC          : {auc:.4f}")
    print(f"  Accuracy @ 0.5   : {acc:.4f}  ({int(acc*len(ys))}/{len(ys)})")
    print(f"  REAL  precision  : {p_real:.4f}    recall: {r_real:.4f}")
    print(f"  FAKE  precision  : {p_fake:.4f}    recall: {r_fake:.4f}")
    print("=" * 60)

    Path("test_result.json").write_text(json.dumps({
        "auc": auc, "accuracy": acc,
        "n_test": int(len(ys)),
        "real": {"precision": p_real, "recall": r_real},
        "fake": {"precision": p_fake, "recall": r_fake},
    }, indent=2))
    print("Saved test_result.json")


if __name__ == "__main__":
    main()
